# Notebook 3 of 4: Schema Drift Demo (Pipeline-Triggered)

Triggered by the Fabric Pipeline `SchemaDriftDemo`.
The pipeline injects parameter values into the cell below at runtime.

To run manually, edit the parameter values and run all cells.

In [ ]:
# Parameters (Fabric pipeline overrides these at runtime)
drift_type     = "column_added"   # column_added | column_removed | type_changed
entity         = "transactions"   # transactions | accounts | customers
force_critical = False             # True -> demo CRITICAL path
run_id         = ""               # auto-generated if blank
openai_api_key = ""               # injected by pipeline as SecureString

In [ ]:
%run ./01_config

In [ ]:
import re
import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

from src.schema_drift_config import (
    SCHEMA_DRIFT_SPEC, ENTITIES, RAW_TABLE_BY_ENTITY,
    LH_TABLE, LAKEHOUSE_NAME, set_ai_key,
)
from src.injector import inject_drift
from src.detector import detect_schema_drift
from src.healer import apply_healing
from src.ai_summary import log_run

spark = SparkSession.builder.getOrCreate()

# Resolve run_id
if not run_id:
    run_id = datetime.now().strftime('run_%Y%m%d_%H%M%S')
else:
    run_id = 'run_' + re.sub(r'[^A-Za-z0-9_]', '_', str(run_id))[:40]

# Pass API key if pipeline injected it
if openai_api_key:
    set_ai_key(openai_api_key)

print('=' * 72)
print(f'  SCHEMA DRIFT DEMO — run_id={run_id}')
print(f'  drift_type   : {drift_type}')
print(f'  entity       : {entity}')
print(f'  force_critical: {force_critical}')
print('=' * 72)

assert drift_type in SCHEMA_DRIFT_SPEC, f'Invalid drift_type: {drift_type}'
assert entity in ENTITIES, f'Invalid entity: {entity}'

In [ ]:
# [1/4] INJECT -> [2/4] DETECT -> [3/4] HEAL
drifted_df, drifted_table, _ = inject_drift(
    spark, entity, drift_type, force_critical, run_id,
    RAW_TABLE_BY_ENTITY[entity], LAKEHOUSE_NAME,
)

anomalies = detect_schema_drift(spark, drifted_df, entity, run_id)

healed_df, actions_taken, quarantined_count = apply_healing(
    spark, drifted_df, entity, anomalies, run_id, LAKEHOUSE_NAME,
)

if healed_df is not None:
    clean_table = f'{LAKEHOUSE_NAME}.clean_{run_id}'
    (healed_df.withColumn('_run_id', lit(run_id))
              .write.format('delta').mode('overwrite')
              .saveAsTable(clean_table))
    surviving = healed_df.count()
    print(f'\n  Healed batch written -> {clean_table}')
    print(f'  Surviving rows: {surviving:,}  |  Quarantined: {quarantined_count:,}')
else:
    surviving = 0
    print(f'\n  Batch fully quarantined — nothing written to clean.')

In [ ]:
# [4/4] AI SUMMARY + LOGGING
summary_record = log_run(
    spark, anomalies, entity, drift_type, force_critical,
    run_id, quarantined_count, surviving, drifted_table,
)

print('\n' + '=' * 72)
print(f'  RUN COMPLETE — {len(anomalies)} anomaly(s), '
      f'max severity {summary_record["max_severity"]}')
print('=' * 72)

# Pipeline-friendly exit
try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit(json.dumps(summary_record, default=str))
except Exception:
    pass